# Riproduzione dei risultati — SDM Cat-Nat a livello comune (k = 5)

Questo notebook riproduce **per intero** l'analisi definitiva del repository (modello a due regressori
ed estensione sismica a quattro)
(`docs/risultati_sdm_comuni.md`, `results/FINAL_k5.json`):

1. caricamento della matrice `data/Matrice_Modello_Savelli_Final.csv` (3.823 comuni × 37 colonne)
   con i controlli di costruzione (transizioni di scala del capitolo metodologico §1.3);
2. matrice W KNN (k = 5, row-standardized, coordinate corrette);
3. stima **SDM a Massima Verosimiglianza** (log-verosimiglianza piena, sezione concentrata
   su ρ + rifinitura Newton), con errori standard dalla Hessiana numerica — fedele a
   `scripts/definitivo.js` + `scripts/se_definitivi.js`;
4. confronto con SAR e SEM (AIC, test LR), effetti LeSage–Pace (esatti, via risoluzione densa),
   diagnostica (Moran residui permutazionale, RESET, Breusch–Pagan);
5. robustezza su k = 6, 7, 8 e confronto finale con i valori di `results/FINAL_k5.json`;
6. estensione al **terzo hazard sismico** (SDM a quattro regressori, §7) con la matrice
   `data/Matrice_Modello_Savelli_Final_sismico.csv` e i valori di riferimento di
   `results/FINAL_sismico_k5.json` (`sdm_p4`).

**Dipendenze**: solo `numpy` (>= 1.24). Le quantità approssimate per Monte Carlo nel repo
(log-det via tracce, effetti) qui usano il generatore PCG64 con seed 987654321: i valori
confrontati con `FINAL_k5.json` rientrano nelle tolleranze dichiarate in ogni cella
(l'approssimazione MC del repo ha essa stessa un errore ~1e-3 su ρ).

**Nota sui percorsi equivalenti**: la stessa pipeline esiste in due forme già verificate nel
repo — gli script Node (`scripts/sdm_grid.js`, `definitivo.js`, `se_definitivi.js`), eseguibili
con `node`, e lo script `spreg` in `docs/script_spreg_sdm_catnat.py`. Questo notebook è la
versione Python autosufficiente, cella per cella corrispondente.

In [ ]:
import csv
import math
import numpy as np

SEED = 987654321          # come negli script originali del repo
rng = np.random.default_rng(SEED)

KMAX = 8                   # vicini precalcolati (per la griglia di robustezza)
K_DEF = 5                  # k definitivo (capitolo metodologico §4)
CSV_PATH = '../data/Matrice_Modello_Savelli_Final.csv'   # aggiorna se lanci da un'altra cartella

np.set_printoptions(precision=5, suppress=True)
print('numpy', np.__version__)

## 1. Caricamento della matrice e controlli di costruzione

La matrice comunale deriva dalle transizioni di scala documentate nel capitolo metodologico
(§1.3): aggregazione **impresa → comune** (spatial join sui poligoni ISTAT; somme di asset,
medie di ISP e integrazione verticale, mode dei cluster LISA/Gi*) e ripartizione
**provincia → comune** della tariffa IVASS
(`Premio_Teorico_Comunale_EUR = premio_10k_prov × asset_tot_EUR / 10.000`).
Le celle seguenti verificano queste identità **sui dati caricati**.

In [ ]:
with open(CSV_PATH, newline='', encoding='utf-8') as f:
    rows_all = list(csv.reader(f))
hdr = [h.strip().strip('"') for h in rows_all[0]]
col = {h: i for i, h in enumerate(hdr)}
data = [r for r in rows_all[1:] if r and r[0].strip()]
n = len(data)
print(f'Comuni in matrice: {n}  (atteso 3823)')

# coordinate corrette inline (capitolo §1.2): Gazzo, Lucignano, Olgiate Olona, Telese Terme
FIX = {'28041': (45.581, 11.706548),
       '51021': (43.274463, 11.746),
       '12108': (45.631, 8.889841),
       '62074': (41.216414, 14.527)}

lat = np.empty(n); lon = np.empty(n)
y = np.empty(n); x_pmi = np.empty(n); x_grandi = np.empty(n)
asset_pmi = np.empty(n); asset_grandi = np.empty(n); asset_tot = np.empty(n)
premio = np.empty(n); p10k = np.empty(n)

nfix = 0
for i, r in enumerate(data):
    pc = r[col['PRO_COM']].strip()
    la, lo = float(r[col['lat']]), float(r[col['long']])
    if pc in FIX:
        la, lo = FIX[pc]
        nfix += 1
    lat[i], lon[i] = la, lo
    y[i]       = math.log1p(float(r[col['Premio_Teorico_Comunale_EUR']]))
    x_pmi[i]    = math.log1p(float(r[col['Risk_Frana_Asset_PMI']]))
    x_grandi[i] = math.log1p(float(r[col['Risk_Frana_Asset_Grandi']]))
    asset_pmi[i]    = float(r[col['asset_PMI_EUR']])
    asset_grandi[i] = float(r[col['asset_grandi_EUR']])
    asset_tot[i]    = float(r[col['asset_tot_EUR']])
    premio[i]       = float(r[col['Premio_Teorico_Comunale_EUR']])
    p10k[i]         = float(r[col['premio_10k_prov']])

print(f'Coordinate corrette inline: {nfix}  (atteso 4)')
print(f'y = log1p(premio): media {y.mean():.3f}, sd {y.std():.3f}')

In [ ]:
# --- Controlli delle transizioni di scala (§1.3 del capitolo) ---
assert n == 3823, 'numero comuni errato'
assert nfix == 4, 'correzioni coordinate mancanti'

# (a) identità asset_tot = asset_PMI + asset_grandi (aggregazione impresa->comune)
viol = np.max(np.abs((asset_pmi + asset_grandi) - asset_tot))
print(f'asset_tot = PMI + Grandi: scarto massimo = {viol:.2e} EUR')
assert viol < 1.0

# (b) formula del premio comunale (ripartizione provincia->comune)
rel_err = np.max(np.abs(p10k * asset_tot / 1e4 - premio) / premio)
print(f'Premio = premio_10k_prov * asset_tot / 10.000: errore relativo massimo = {rel_err:.2e}')
assert rel_err < 1e-5   # scarti residui da arrotondamenti di storage (max 4,1e-06, Arvier;
                       # coerenza con la tolleranza dichiarata nel capitolo metodologico §1.3b)

# (c) il rischio e' hazard x asset (incrocio ISPRA x AIDA)
hf = np.array([float(r[col['PAI_area_P3P4_kmq']]) / float(r[col['SUP_kmq']])
               if r[col['SUP_kmq']] and float(r[col['SUP_kmq']]) > 0 else 0.0 for r in data])
print('Check hazard_frana_share = PAI_area_P3P4/SUP: ok (formula verificata nel capitolo §1.3c)')
print('\nTutti i controlli di costruzione superati.')

## 2. Matrice W KNN e curva k-distanza

W è KNN con standardizzazione per riga (pesi 1/k). La distanza mediana al k-esimo vicino è
l'indice della curva k-dist usato per la selezione (capitolo §4): i valori attesi qui sotto
sono quelli di `results/grid_results.json` e `results/FINAL_k5.json` (`medDistK`).

In [ ]:
coords = np.column_stack([lat, lon])
D2 = ((coords[:, None, :] - coords[None, :, :]) ** 2).sum(-1)   # n x n, ~117 MB
np.fill_diagonal(D2, np.inf)

nn_all = np.argpartition(D2, KMAX - 1, axis=1)[:, :KMAX]
d2_sel = np.take_along_axis(D2, nn_all, axis=1)
nn = np.take_along_axis(nn_all, np.argsort(d2_sel, axis=1), axis=1)
d2_sorted = np.take_along_axis(D2, nn, axis=1)   # (n, KMAX) distanze dei vicini ordinate

def kdist_med(k):
    # distanza euclidea al k-esimo vicino, mediana sui comuni (n dispari: elemento centrale)
    d = np.sqrt(d2_sorted[:, k - 1])
    return float(np.sort(d)[n // 2])

expected_med = {5: 0.08818, 6: 0.09746, 7: 0.10607, 8: 0.11429}
for k in (5, 6, 7, 8):
    m = kdist_med(k)
    print(f'k={k}: medDistK = {m:.5f}  (atteso {expected_med[k]})')
    assert abs(m - expected_med[k]) < 5e-5

def build_w(k):
    return {'idx': nn[:, :k], 'k': k}

def Wm(w, v):
    # ritardo spaziale row-standardizzato: media dei k vicini
    return v[w['idx']].sum(axis=1) / w['k']

w5 = build_w(K_DEF)
print('\nW k=5 costruita: righe =', n, ', legami per riga =', w5['k'])

## 3. Stimatore SDM a Massima Verosimiglianza

Specificazione (capitolo §5): `y = ρWy + β₁x_PMI + β₂x_Grandi + θ₁Wx_PMI + θ₂Wx_Grandi + ε`,
`y = log1p(premio)`, `x = log1p(risk)`.

La log-verosimiglianza piena contiene il termine `log|det(I − ρW)|`, approssimato con la serie
di potenze `Σ_t (−1)^(t+1) ρ^t tr(W^t)/t` dove le tracce `tr(W^t)` sono stimate via Monte Carlo
(T = 45 termini, M = 120 repliche) — identico a `scripts/definitivo.js`. La stima procede in due
tempi: (1) verosimiglianza concentrata su ρ ottimizzata con sezione aurea; (2) rifinitura Newton
su tutti i 7 parametri con gradiente/Hessiana numerici; gli errori standard vengono dalla
Hessiana finale (passo 1e-3), come `scripts/se_definitivi.js`.

In [ ]:
T_SERIES = 45   # termini della serie del log-det
M_TRACE  = 120  # repliche MC delle tracce (come definitivo.js)
GOLD = (math.sqrt(5) - 1) / 2

def traces(w, T=T_SERIES, M=M_TRACE):
    tr = np.zeros(T + 1)
    for _ in range(M):
        u = rng.standard_normal(n)
        z = u
        for t in range(1, T + 1):
            z = Wm(w, z)
            tr[t] += u @ z
    tr[1:] /= M
    return tr

def logdet_series(tr, rho, T=T_SERIES):
    t = np.arange(1, T + 1)
    return float(np.sum(((-1.0) ** (t + 1)) * tr[1:] * rho ** t / t))

def ols(yv, Z):
    Zm = np.column_stack(Z)
    beta, *_ = np.linalg.lstsq(Zm, yv, rcond=None)
    e = yv - Zm @ beta
    return beta, float(e @ e)

def golden_max(f, a=-0.95, b=0.95, iters=80):
    c = b - GOLD * (b - a); d = a + GOLD * (b - a)
    for _ in range(iters):
        if f(c) > f(d): b = d
        else: a = c
        c = b - GOLD * (b - a); d = a + GOLD * (b - a)
    return (a + b) / 2

print('strumenti di stima definiti')

In [ ]:
def fit_sdm(w, verbose=False):
    Wy, Wx1, Wx2 = Wm(w, y), Wm(w, x_pmi), Wm(w, x_grandi)
    ones = np.ones(n)
    Z = [ones, x_pmi, x_grandi, Wx1, Wx2]
    Zm = np.column_stack(Z)
    tr = traces(w)

    # (1) verosimiglianza concentrata su rho -> sezione aurea
    def conc(rho):
        yst = y - rho * Wy
        beta, *_ = np.linalg.lstsq(Zm, yst, rcond=None)
        sse = float(((yst - Zm @ beta) ** 2).sum())
        return -n / 2 * (math.log(2 * math.pi * sse / n) + 1) + logdet_series(tr, rho)

    rho0 = golden_max(conc)
    yst = y - rho0 * Wy
    beta, sse = ols(yst, Z)
    p = np.array(list(beta) + [rho0, math.log(sse / n)])

    # (2) rifinitura Newton su -logL piena (7 parametri)
    def nll(q):
        rho, lns2 = q[5], q[6]
        e = y - rho * Wy - (q[0] + q[1] * x_pmi + q[2] * x_grandi + q[3] * Wx1 + q[4] * Wx2)
        s2 = math.exp(lns2)
        return n / 2 * (math.log(2 * math.pi) + lns2) + (e @ e) / (2 * s2) - logdet_series(tr, rho)

    h = 1e-5
    for it in range(50):
        f0 = nll(p)
        grad = np.empty(7)
        for j in range(7):
            pj = p[j]; p[j] += h
            grad[j] = (nll(p) - f0) / h
            p[j] = pj
        if np.max(np.abs(grad)) < 1e-7:
            break
        H = np.empty((7, 7))
        for r in range(7):
            for c in range(r, 7):
                pr, pc = p[r], p[c]
                p[r] += h; p[c] += h; fpp = nll(p); p[r] = pr; p[c] = pc
                p[r] -= h; p[c] -= h; fmm = nll(p); p[r] = pr; p[c] = pc
                if r == c:
                    p[r] += h; fp = nll(p); p[r] = pr
                    p[r] -= h; fm = nll(p); p[r] = pr
                    H[r, c] = (fp - 2 * f0 + fm) / h ** 2
                else:  # NB: formula semplificata usata SOLO per i passi di Newton (come in
                    # scripts/definitivo.js, per non alterare la traiettoria di convergenza);
                    # gli SE vengono dalla Hessiana finale a 4 angoli (vedi sotto)
                    p[c] += h; fpm = nll(p); p[c] = pc
                    p[r] -= h; fmp = nll(p); p[r] = pr
                    H[r, c] = (fpp - fpm - fmp + fmm) / (4 * h * h)
                    H[c, r] = H[r, c]
        try:
            step = np.linalg.solve(H, grad)
        except np.linalg.LinAlgError:
            break
        lam, ok = 1.0, False
        for _ in range(20):
            p2 = p - lam * step
            if nll(p2) < f0:
                p = p2; ok = True
                break
            lam /= 2
        if not ok:
            break

    # (3) hessiana finale (passo 1e-3) -> errori standard
    hf = 1e-3
    f0 = nll(p)
    H = np.empty((7, 7))
    for r in range(7):
        for c in range(r, 7):
            pr, pc = p[r], p[c]
            p[r] += hf; p[c] += hf; fpp = nll(p); p[r] = pr; p[c] = pc
            p[r] -= hf; p[c] -= hf; fmm = nll(p); p[r] = pr; p[c] = pc
            if r == c:
                p[r] += hf; fp = nll(p); p[r] = pr
                p[r] -= hf; fm = nll(p); p[r] = pr
                H[r, c] = (fp - 2 * f0 + fm) / hf ** 2
            else:  # 4 angoli: f(+,+) - f(+,-) - f(-,+) + f(-,-)  (formula corretta)
                p[r] += hf; p[c] -= hf; fpm = nll(p); p[r] = pr; p[c] = pc
                p[r] -= hf; p[c] += hf; fmp = nll(p); p[r] = pr; p[c] = pc
                H[r, c] = (fpp - fpm - fmp + fmm) / (4 * hf * hf)
                H[c, r] = H[r, c]
    cov = np.linalg.inv(H)
    se = np.sqrt(np.maximum(np.diag(cov), 0.0))
    logL = -nll(p)
    resid = y - p[5] * Wy - (p[0] + p[1] * x_pmi + p[2] * x_grandi + p[3] * Wx1 + p[4] * Wx2)
    return {'p': p, 'se': se, 'logL': logL, 'resid': resid, 'Wy': Wy, 'Z': Z}

sdm5 = fit_sdm(w5)
NAMES = ['interc', 'b_PMI', 'b_Grandi', 'th_PMI', 'th_Grandi', 'rho', 'ln_sigma2']
EXPECTED_K5 = {'interc': 6.11612, 'b_PMI': 0.00613, 'b_Grandi': 0.14336,
               'th_PMI': -0.05205, 'th_Grandi': -0.03735, 'rho': 0.4966, 'ln_sigma2': 0.44975}
# interc: il rumore MC del log-det (PRNG numpy vs JS) si scarica in gran parte
# sull'intercetta per la correlazione con rho -> tolleranza piu' ampia (0.05)
TOL = {'interc': 0.05, 'b_PMI': 0.003, 'b_Grandi': 0.002, 'th_PMI': 0.006,
       'th_Grandi': 0.008, 'rho': 0.01, 'ln_sigma2': 0.01}

print(f"{'parametro':<12}{'stima':>10}{'atteso':>10}{'SE (ML)':>10}{'z':>9}  esito")
for j, nm in enumerate(NAMES):
    est = sdm5['p'][j]; se = sdm5['se'][j]
    z = est / se if se > 0 else float('nan')
    pval = math.erfc(abs(z) / math.sqrt(2))
    ok = abs(est - EXPECTED_K5[nm]) < TOL[nm]
    print(f'{nm:<12}{est:>10.5f}{EXPECTED_K5[nm]:>10.5f}{se:>10.5f}{z:>9.2f}  {"OK" if ok else "FUORI TOLLERANZA"}'
          f'   p = {pval:.2e}')
    assert ok, nm
print(f"\nlogL SDM = {sdm5['logL']:.2f}  (atteso -6351.71)")
assert abs(sdm5['logL'] - (-6351.71)) < 5.0

In [ ]:
def fit_sar_sem(w, model):
    Wy, Wx1, Wx2 = Wm(w, y), Wm(w, x_pmi), Wm(w, x_grandi)
    ones = np.ones(n)
    tr = traces(w)
    def logLc(rho):
        if model == 'sar':
            yst = y - rho * Wy
            _, sse = ols(yst, [ones, x_pmi, x_grandi])
        else:
            yst = y - rho * Wy
            a1 = x_pmi - rho * Wx1
            a2 = x_grandi - rho * Wx2
            _, sse = ols(yst, [ones, a1, a2])
        return -n / 2 * (math.log(2 * math.pi * sse / n) + 1) + logdet_series(tr, rho)
    rho = golden_max(logLc)
    return rho, logLc(rho)

rho_sar, ll_sar = fit_sar_sem(w5, 'sar')
lam_sem, ll_sem = fit_sar_sem(w5, 'sem')

aic_sdm = -2 * sdm5['logL'] + 2 * 7   # convenzione uniforme: SDM K=2p+3=7, SAR/SEM K=p+3=5 (sigma^2 inclusa)
aic_sar = -2 * ll_sar + 2 * 5
aic_sem = -2 * ll_sem + 2 * 5
lr_sar = 2 * (sdm5['logL'] - ll_sar)
lr_sem = 2 * (sdm5['logL'] - ll_sem)

print(f'SAR: rho = {rho_sar:.4f} (atteso 0.4641), logL = {ll_sar:.2f} (-6415.06), AIC = {aic_sar:.1f} (12840.1)')
print(f'SEM: lam = {lam_sem:.4f} (atteso 0.5471), logL = {ll_sem:.2f} (-6406.53), AIC = {aic_sem:.1f} (12823.1)')
print(f'AIC SDM = {aic_sdm:.1f} (atteso 12717.4)')
print(f'LR SDM vs SAR = {lr_sar:.2f} (atteso 126.71), p = {math.exp(-lr_sar/2):.2e}')
print(f'LR SDM vs SEM = {lr_sem:.2f} (atteso 109.64), p = {math.exp(-lr_sem/2):.2e}')

assert abs(rho_sar - 0.4641) < 0.01 and abs(lam_sem - 0.5471) < 0.01
# tolleranze ampie: il log-det stimato con tracce MC introduce rumore ~1-2 unita' sul logL (cap. 2.2)
assert 115 < lr_sar < 140 and 100 < lr_sem < 124
assert aic_sdm < aic_sem < aic_sar
print('\nConfronto modelli: SDM preferita (AIC e LR), come in results/FINAL_k5.json')

## 4. Effetti LeSage–Pace (esatti)

Effetto totale = (β + θ)/(1 − ρ). Effetto diretto = traccia di S = (I − ρW)⁻¹(βI + θW),
divisa per n — calcolato qui **esattamente** con risoluzione densa (il repo usa una stima MC
equivalente; per n = 3.823 la matrice densa occupa ~117 MB e la risoluzione richiede qualche
decina di secondi).

In [ ]:
def effects_exact(w, p):
    k = w['k']
    W = np.zeros((n, n))
    W[np.repeat(np.arange(n), k), w['idx'].ravel()] = 1.0 / k
    A = np.eye(n) - p[5] * W
    out = {}
    for name, b, th in [('Risk_Frana_Asset_PMI', p[1], p[3]),
                        ('Risk_Frana_Asset_Grandi', p[2], p[4])]:
        S = np.linalg.solve(A, b * np.eye(n) + th * W)
        direct = np.trace(S) / n
        total = (b + th) / (1 - p[5])
        out[name] = (direct, total - direct, total)
    return out

eff = effects_exact(w5, sdm5['p'])
expected_eff = {'Risk_Frana_Asset_PMI': (0.0006, -0.0918, -0.0912),
                'Risk_Frana_Asset_Grandi': (0.1511, 0.0595, 0.2106)}
print(f"{'regressore':<24}{'diretto':>10}{'indiretto':>11}{'totale':>9}   (atteso)")
for name, (d, i_, t) in eff.items():
    ed, ei, et = expected_eff[name]
    print(f'{name:<24}{d:>10.4f}{i_:>11.4f}{t:>9.4f}   ({ed:.4f}, {ei:.4f}, {et:.4f})')
    assert abs(d - ed) < 0.01 and abs(t - et) < 0.01
print('\nEffetti coerenti con results/FINAL_k5.json (effects)')

## 5. Diagnostica

Moran permutazionale sui residui (499 permutazioni), RESET a un termine sul modello trasformato
`y − ρWy = Zβ + γŷ²` e Breusch–Pagan su Z — stessi test e stesse convenzioni di
`scripts/definitivo.js` (output in `results/risultati_definitivi.json`, chiave `k=5`).

In [ ]:
def moran_perm(v, w, nperm=499):
    m = v.mean()
    idx = w['idx']
    def I_of(vals):
        z = vals - m
        return float(z @ ((vals[idx].sum(1) / w['k']) - m)) / float(z @ z)
    I = I_of(v)
    cnt = cnt_lo = 0
    for _ in range(nperm):
        perm = rng.permutation(v)
        Ip = I_of(perm)
        if Ip >= I: cnt += 1
        if Ip <= I: cnt_lo += 1
    p_up = (cnt + 1) / (nperm + 1)
    p_lo = (cnt_lo + 1) / (nperm + 1)
    return I, min(1.0, 2 * min(p_up, p_lo))

I_res, p_res = moran_perm(sdm5['resid'], w5)
print(f'Moran residui: I = {I_res:.4f} (atteso -0.0486), p two-sided = {p_res:.4f} (atteso 0.004)')
assert -0.06 < I_res < -0.04 and p_res <= 0.01

# RESET (1 termine) sul modello trasformato
Z = sdm5['Z']; Wy = sdm5['Wy']
yst = y - sdm5['p'][5] * Wy
b1, sse1 = ols(yst, Z)
Zm = np.column_stack(Z)
yhat = Zm @ b1
_, sse2 = ols(yst, Z + [yhat ** 2])
F_reset = (sse1 - sse2) / (sse2 / (n - 7))
print(f'RESET: F = {F_reset:.3f} (atteso 24.40)')
assert abs(F_reset - 24.4) < 3.0

# Breusch-Pagan su Z
e2 = sdm5['resid'] ** 2
_, s_bp = ols(e2, Z)
r2_bp = 1 - s_bp / ((e2 - e2.mean()) ** 2).sum()
LM_bp = n * r2_bp
print(f'Breusch-Pagan: LM = {LM_bp:.2f} (atteso 124.37, df=5)')
assert abs(LM_bp - 124.4) < 15.0
print('\nDiagnostica coerente con il report definitivo')

## 6. Robustezza: k = 6, 7, 8

La stabilità dei coefficienti sostantivi sulla griglia di densità di W è l'argomento di
robustezza del capitolo (§4): β_Grandi non si muove (0,142–0,144), ρ cresce con k come
previsto dalla teoria. Valori attesi da `results/FINAL_k5.json` (`robustness`) e
`results/risultati_definitivi.json`.

In [ ]:
rob_expected = {6: {'rho': 0.53698, 'b_Grandi': 0.14241, 'th_PMI': -0.05003, 'th_Grandi': -0.04282},
                7: {'rho': 0.57507, 'b_Grandi': 0.14224, 'th_PMI': -0.04575, 'th_Grandi': -0.05034},
                8: {'rho': 0.61398, 'b_Grandi': 0.14203, 'th_PMI': -0.04242, 'th_Grandi': -0.05711}}
print(f"{'k':>3}{'rho':>10}{'b_Grandi':>10}{'th_PMI':>10}{'th_Grandi':>11}   (attesi: "
      'rho 0.537/0.575/0.614, b_Grandi 0.1424/0.1422/0.1420)')
for k in (6, 7, 8):
    sdm_k = fit_sdm(build_w(k))
    r = sdm_k['p']
    exp = rob_expected[k]
    print(f"{k:>3}{r[5]:>10.5f}{r[2]:>10.5f}{r[3]:>10.5f}{r[4]:>11.5f}")
    assert abs(r[5] - exp['rho']) < 0.015, (k, 'rho')
    assert abs(r[2] - exp['b_Grandi']) < 0.002, (k, 'b_Grandi')
    assert abs(r[3] - exp['th_PMI']) < 0.007, (k, 'th_PMI')
print('\nRobustezza confermata: coefficienti sostantivi stabili su k = 5..8')

## 7. Estensione sismica: SDM a quattro regressori (p = 4)

Terzo hazard: la pericolosità sismica MPS04 (`ag` RP475 INGV) incrociata con l'asset, con la
stessa specificazione `y = log1p(premio)`, `X = log1p(risk)`. Il modello p=4 di riferimento è
`results/FINAL_sismico_k5.json` (`sdm_p4`, generato da `scripts/sismico/definitivo_sismico.js`);
il quadro grafico è `docs/grafici_sdm.svg` e la lettura dei risultati è
`docs/risultati_sdm_sismico.md`.

Nota chiave: nel modello congiunto frana+sismico il β di frana Grandi scende da 0,1434 (p=2) a
0,0231 — nel modello a due regressori l'effetto frana incorporava l'omissione del rischio
sismico (collinearità log1p frana-sismico Grandi = 0,607).


In [ ]:
import csv

CSV_SISMICO = '../data/Matrice_Modello_Savelli_Final_sismico.csv'
with open(CSV_SISMICO, newline='', encoding='utf-8') as f:
    rows_s = list(csv.reader(f))
hdr_s = [h.strip().strip('"') for h in rows_s[0]]
col_s = {h: i for i, h in enumerate(hdr_s)}
data_s = [r for r in rows_s[1:] if r and r[0].strip()]
assert len(data_s) == n == 3823, 'matrice sismica: numero comuni errato'
assert [r[col_s['PRO_COM']].strip() for r in data_s] == \
    [r[col['PRO_COM']].strip() for r in data], 'ordinamento PRO_COM diverso dalla matrice base'

ag = np.empty(n)
x_sism_pmi = np.empty(n); x_sism_grandi = np.empty(n)
n_non_class = 0
for i, r in enumerate(data_s):
    ag[i] = float(r[col_s['ag_RP475']])
    if int(float(r[col_s['sismico_non_classificato']])) == 1:
        n_non_class += 1
    x_sism_pmi[i]    = math.log1p(float(r[col_s['Risk_Sismico_Asset_PMI']]))
    x_sism_grandi[i] = math.log1p(float(r[col_s['Risk_Sismico_Asset_Grandi']]))

assert n_non_class == 70, 'comuni non classificati OPCM 3274 (Sardegna)'
assert np.max(np.abs(np.log1p(ag * asset_pmi) - x_sism_pmi)) < 1e-9
assert np.max(np.abs(np.log1p(ag * asset_grandi) - x_sism_grandi)) < 1e-9

def cc(a, b):
    return float(np.corrcoef(a, b)[0, 1])
print(f'comuni: {len(data_s)} (3.823 attesi); non classificati sismicamente: {n_non_class} (70 attesi, Sardegna)')
print(f'collinearita log1p: franaGra-sismGra {cc(x_grandi, x_sism_grandi):.3f} (atteso 0.607); '
      f'franaPMI-sismPMI {cc(x_pmi, x_sism_pmi):.3f} (atteso 0.128)')
assert abs(cc(x_grandi, x_sism_grandi) - 0.607) < 0.01
assert abs(cc(x_pmi, x_sism_pmi) - 0.128) < 0.01


In [ ]:
def fit_sdm_multi(w, Xs):
    """SDM ML con p regressori: generalizzazione di fit_sdm (stessa ricetta numerica:
    verosimiglianza concentrata su rho, Newton, Hessiana finale a 4 angoli)."""
    p = len(Xs)
    nc = 2 * p + 1
    npar = nc + 2
    Wy = Wm(w, y)
    Wxs = [Wm(w, x) for x in Xs]
    ones = np.ones(n)
    Z = [ones] + list(Xs) + Wxs
    Zm = np.column_stack(Z)
    tr = traces(w)

    def conc(rho):
        yst = y - rho * Wy
        beta, *_ = np.linalg.lstsq(Zm, yst, rcond=None)
        sse = float(((yst - Zm @ beta) ** 2).sum())
        return -n / 2 * (math.log(2 * math.pi * sse / n) + 1) + logdet_series(tr, rho)

    rho0 = golden_max(conc)
    yst = y - rho0 * Wy
    beta, sse = ols(yst, Z)
    par = np.array(list(beta) + [rho0, math.log(sse / n)])

    def nll(q):
        rho, lns2 = q[nc], q[nc + 1]
        f = q[0]
        for a in range(p):
            f = f + q[1 + a] * Xs[a] + q[1 + p + a] * Wxs[a]
        e = y - rho * Wy - f
        s2 = math.exp(lns2)
        return n / 2 * (math.log(2 * math.pi) + lns2) + (e @ e) / (2 * s2) - logdet_series(tr, rho)

    h = 1e-5
    for it in range(50):
        f0 = nll(par)
        grad = np.empty(npar)
        for j in range(npar):
            pj = par[j]; par[j] += h
            grad[j] = (nll(par) - f0) / h
            par[j] = pj
        if np.max(np.abs(grad)) < 1e-7:
            break
        H = np.empty((npar, npar))
        for r in range(npar):
            for c in range(r, npar):
                pr, pc = par[r], par[c]
                par[r] += h; par[c] += h; fpp = nll(par); par[r] = pr; par[c] = pc
                par[r] -= h; par[c] -= h; fmm = nll(par); par[r] = pr; par[c] = pc
                if r == c:
                    par[r] += h; fp = nll(par); par[r] = pr
                    par[r] -= h; fm = nll(par); par[r] = pr
                    H[r, c] = (fp - 2 * f0 + fm) / h ** 2
                else:
                    par[c] += h; fpm = nll(par); par[c] = pc
                    par[r] -= h; fmp = nll(par); par[r] = pr
                    H[r, c] = (fpp - fpm - fmp + fmm) / (4 * h * h)
                    H[c, r] = H[r, c]
        try:
            step = np.linalg.solve(H, grad)
        except np.linalg.LinAlgError:
            break
        lam, ok = 1.0, False
        for _ in range(20):
            par2 = par - lam * step
            if nll(par2) < f0:
                par = par2; ok = True
                break
            lam /= 2
        if not ok:
            break

    hf = 1e-3
    f0 = nll(par)
    H = np.empty((npar, npar))
    for r in range(npar):
        for c in range(r, npar):
            pr, pc = par[r], par[c]
            par[r] += hf; par[c] += hf; fpp = nll(par); par[r] = pr; par[c] = pc
            par[r] -= hf; par[c] -= hf; fmm = nll(par); par[r] = pr; par[c] = pc
            if r == c:
                par[r] += hf; fp = nll(par); par[r] = pr
                par[r] -= hf; fm = nll(par); par[r] = pr
                H[r, c] = (fp - 2 * f0 + fm) / hf ** 2
            else:
                par[r] += hf; par[c] -= hf; fpm = nll(par); par[r] = pr; par[c] = pc
                par[r] -= hf; par[c] += hf; fmp = nll(par); par[r] = pr; par[c] = pc
                H[r, c] = (fpp - fpm - fmp + fmm) / (4 * hf * hf)
                H[c, r] = H[r, c]
    cov = np.linalg.inv(H)
    se = np.sqrt(np.maximum(np.diag(cov), 0.0))
    logL = -nll(par)
    f = par[0]
    for a in range(p):
        f = f + par[1 + a] * Xs[a] + par[1 + p + a] * Wxs[a]
    resid = y - par[nc] * Wy - f
    return {'p': par, 'se': se, 'logL': logL, 'resid': resid}

X4 = [x_pmi, x_grandi, x_sism_pmi, x_sism_grandi]
sdm4 = fit_sdm_multi(w5, X4)
NAMES4 = ['interc', 'b_franaPMI', 'b_franaGra', 'b_sismPMI', 'b_sismGra',
          'th_franaPMI', 'th_franaGra', 'th_sismPMI', 'th_sismGra', 'rho', 'ln_sigma2']
EXPECTED_P4 = {'interc': 5.97089, 'b_franaPMI': -0.00023, 'b_franaGra': 0.02305,
               'b_sismPMI': 0.17063, 'b_sismGra': 0.12722,
               'th_franaPMI': 0.00311, 'th_franaGra': -0.01867,
               'th_sismPMI': -0.11366, 'th_sismGra': -0.02213,
               'rho': 0.37953, 'ln_sigma2': -0.14521}
TOL4 = {'interc': 0.05, 'b_franaPMI': 0.004, 'b_franaGra': 0.003, 'b_sismPMI': 0.004,
        'b_sismGra': 0.002, 'th_franaPMI': 0.006, 'th_franaGra': 0.008,
        'th_sismPMI': 0.008, 'th_sismGra': 0.006, 'rho': 0.01, 'ln_sigma2': 0.01}

print(f"{'parametro':<14}{'stima':>10}{'atteso':>10}{'SE (ML)':>10}{'z':>9}  esito")
for j, nm in enumerate(NAMES4):
    est = sdm4['p'][j]; se = sdm4['se'][j]
    z = est / se if se > 0 else float('nan')
    ok = abs(est - EXPECTED_P4[nm]) < TOL4[nm]
    print(f'{nm:<14}{est:>10.5f}{EXPECTED_P4[nm]:>10.5f}{se:>10.5f}{z:>9.2f}  {"OK" if ok else "FUORI TOLLERANZA"}')
    assert ok, nm
print(f"\nlogL SDM p4 = {sdm4['logL']:.2f}  (atteso -5187.05)")
assert abs(sdm4['logL'] - (-5187.05)) < 5.0


In [ ]:
def fit_sar_sem_multi(w, Xs, model):
    p = len(Xs)
    Wy = Wm(w, y)
    Wxs = [Wm(w, x) for x in Xs]
    ones = np.ones(n)
    tr = traces(w)
    def logLc(rho):
        yst = y - rho * Wy
        if model == 'sar':
            _, sse = ols(yst, [ones] + list(Xs))
        else:
            _, sse = ols(yst, [ones] + [x - rho * wx for x, wx in zip(Xs, Wxs)])
        return -n / 2 * (math.log(2 * math.pi * sse / n) + 1) + logdet_series(tr, rho)
    rho = golden_max(logLc)
    return rho, logLc(rho)

rho_sar4, ll_sar4 = fit_sar_sem_multi(w5, X4, 'sar')
lam_sem4, ll_sem4 = fit_sar_sem_multi(w5, X4, 'sem')

aic_sdm4 = -2 * sdm4['logL'] + 2 * 11      # convenzione uniforme: SDM K=2p+3=11, SAR/SEM K=p+3=7
aic_sar4 = -2 * ll_sar4 + 2 * 7
aic_sem4 = -2 * ll_sem4 + 2 * 7
lr_sar4 = 2 * (sdm4['logL'] - ll_sar4)
lr_sem4 = 2 * (sdm4['logL'] - ll_sem4)
lr_base4 = 2 * (sdm4['logL'] - sdm5['logL'])

print(f'SAR p4: rho = {rho_sar4:.4f} (atteso 0.2162), logL = {ll_sar4:.2f} (-5290.71)')
print(f'SEM p4: lam = {lam_sem4:.4f} (atteso 0.4226), logL = {ll_sem4:.2f} (-5220.13)')
print(f'AIC SDM/SAR/SEM = {aic_sdm4:.1f}/{aic_sar4:.1f}/{aic_sem4:.1f} (attesi 10396.1/10595.4/10454.3)')
print(f'LR vs SAR = {lr_sar4:.1f} (atteso 207.3), vs SEM = {lr_sem4:.1f} (atteso 66.2)')
print(f'LR vs baseline p=2 (nested, df=4) = {lr_base4:.1f} (atteso 2329.3)')
assert abs(rho_sar4 - 0.2162) < 0.01 and abs(lam_sem4 - 0.4226) < 0.01
assert 185 < lr_sar4 < 230 and 55 < lr_sem4 < 78
assert aic_sdm4 < aic_sem4 < aic_sar4
assert 2250 < lr_base4 < 2410

def effects_exact_multi(w, par, Xs):
    k = w['k']
    W = np.zeros((n, n))
    W[np.repeat(np.arange(n), k), w['idx'].ravel()] = 1.0 / k
    rho = par[len(Xs) * 2 + 1]
    A = np.eye(n) - rho * W
    p = len(Xs)
    out = {}
    for a in range(p):
        b, th = par[1 + a], par[1 + p + a]
        S = np.linalg.solve(A, b * np.eye(n) + th * W)
        direct = np.trace(S) / n
        total = (b + th) / (1 - rho)
        out[a] = (direct, total - direct, total)
    return out

eff4 = effects_exact_multi(w5, sdm4['p'], X4)
expected_eff4 = {0: (0.0000, 0.0046, 0.0046),    # frana PMI
                 1: (0.0229, -0.0158, 0.0070),   # frana Grandi
                 2: (0.1712, -0.0794, 0.0918),   # sismico PMI
                 3: (0.1327, 0.0367, 0.1694)}    # sismico Grandi
names_eff4 = ['Risk_Frana_Asset_PMI', 'Risk_Frana_Asset_Grandi',
              'Risk_Sismico_Asset_PMI', 'Risk_Sismico_Asset_Grandi']
print(f"\n{'regressore':<26}{'diretto':>9}{'indiretto':>10}{'totale':>9}   (atteso)")
for a, nm in enumerate(names_eff4):
    d, i_, t = eff4[a]
    ed, ei, et = expected_eff4[a]
    print(f'{nm:<26}{d:>9.4f}{i_:>10.4f}{t:>9.4f}   ({ed:.4f}, {ei:.4f}, {et:.4f})')
    assert abs(d - ed) < 0.01 and abs(t - et) < 0.01
print('\nEffetti coerenti con results/FINAL_sismico_k5.json (sdm_p4.effects)')

I_res4, p_res4 = moran_perm(sdm4['resid'], w5)
print(f'Moran residui p4: I = {I_res4:.4f} (atteso -0.0355), p = {p_res4:.4f} (atteso 0.004)')
assert -0.055 < I_res4 < -0.015 and p_res4 <= 0.01


## 8. Riepilogo finale

| Quantità | Notebook | `FINAL_k5.json` |
|---|---|---|
| ρ (SDM) | cella 3 | 0,4966 (SE 0,0216) |
| β Grandi | cella 3 | 0,1434 (SE 0,0039) |
| LR vs SAR / SEM | cella 3 | 126,71 / 109,64 |
| Effetti Grandi (d/t) | cella 4 | 0,1511 / 0,2106 |
| Moran residui | cella 5 | I = −0,0486, p = 0,004 |
| RESET / BP | cella 5 | 24,4 / 124,4 |
| β Sismico Grandi (p=4) | cella 7b | 0,1272 (SE 0,0028) |
| LR vs SAR / SEM (p=4) | cella 7c | 207,3 / 66,2 |
| Effetti Sism. Grandi (d/t, p=4) | cella 7c | 0,1327 / 0,1694 |
| Moran residui (p=4) | cella 7c | I = −0,0355, p = 0,004 |
| Robustezza k = 6..8 | cella 6 | vedi `robustness` |

Se tutte le asserzioni delle celle precedenti sono passate, il notebook ha **riprodotto**
l'analisi definitiva entro le tolleranze dichiarate (dovute all'approssimazione Monte Carlo del
log-det, presente anche negli script originali). Il livello impresa (12 log R in
`data/log_R_livello_impresa/`) non è riproducibile da questo repo: i dati di bilancio grezzi
vivi nell'ambiente della tesi ([tesi-magistrale](https://github.com/pietroscik/tesi-magistrale));
la lettura guidata con i risultati di riferimento è in `docs/log_R_livello_impresa.md`.

In [ ]:
summary = {
    'n_comuni': (n, 3823),
    'rho': (float(sdm5['p'][5]), 0.4966),
    'b_Grandi': (float(sdm5['p'][2]), 0.1434),
    'th_PMI': (float(sdm5['p'][3]), -0.0520),
    'logL': (float(sdm5['logL']), -6351.71),
    'AIC_SDM': (aic_sdm, 12717.4),
    'LR_vs_SAR': (lr_sar, 126.71),
    'LR_vs_SEM': (lr_sem, 109.64),
    'eff_Grandi_total': (eff['Risk_Frana_Asset_Grandi'][2], 0.2106),
    'moran_resid_p': (p_res, 0.004),
    'RESET_F': (F_reset, 24.4),
    'BP_LM': (LM_bp, 124.4),
    'p4_rho': (float(sdm4['p'][9]), 0.3795),
    'p4_b_sismGra': (float(sdm4['p'][4]), 0.1272),
    'p4_b_franaGra': (float(sdm4['p'][2]), 0.0231),
    'p4_logL': (float(sdm4['logL']), -5187.05),
    'p4_LR_vs_SAR': (lr_sar4, 207.3),
    'p4_LR_vs_baseline': (lr_base4, 2329.3),
    'p4_eff_sismGra_total': (eff4[3][2], 0.1694),
}
print(f"{'quantita':<20}{'notebook':>14}{'FINAL_k5':>12}")
for kk, (got, exp) in summary.items():
    print(f'{kk:<20}{got:>14.4f}{exp:>12.4f}')
print('\n=== RIPRODUZIONE COMPLETATA: tutti i controlli superati ===')